# SVM カーネル（線形 vs RBF）・k 近傍法・ナイーブベイズによる文書分類と根拠分析

書籍 第2章「分類・回帰問題の解き方」（2-1-2 評価極性分析・ナイーブベイズ・SVM カーネル、2-2-2 文書分類の k 近傍法、関連論文 p.213）に対応する演習ノートブックです。
[BERT 演習](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch2_bert.ipynb)・[TF-IDF ベースライン演習](https://colab.research.google.com/github/mamorlis/nlpbook/blob/main/notebooks/ch1_tfidf_baseline.ipynb)と同じ `covid-annotation.csv` を使い、CPU のみ・数秒で以下を比較します。

1. **高次元スパースなテキスト特徴量に対する SVM カーネル比較**（線形 vs 多項式 vs RBF、p.55）
2. **TF-IDF コサイン類似度 + $k$ 近傍法（k-NN）** と、予測根拠となった類似文書の可視化（p.62）
3. **ナイーブベイズ分類器（`MultinomialNB`）とラプラス平滑化**、対数尤度比による特徴語分析（2-1-2・関連論文 p.213）

In [1]:
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/covid-annotation.csv

In [2]:
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC

df = pd.read_csv("covid-annotation.csv").fillna({"コメント": ""})
df["y"] = df["ラベル"].astype(int)

train_dev_df, test_df = train_test_split(df, test_size=0.1, stratify=df["y"], random_state=42)
train_df, dev_df = train_test_split(train_dev_df, test_size=0.1, stratify=train_dev_df["y"], random_state=42)

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2)
X_train = vec.fit_transform(train_df["文書"])
X_dev = vec.transform(dev_df["文書"])
X_test = vec.transform(test_df["文書"])

y_train = train_df["y"].to_numpy()
y_dev = dev_df["y"].to_numpy()
y_test = test_df["y"].to_numpy()

print(f"訓練データ行列の形状: {X_train.shape} (非ゼロ率 = {X_train.nnz / np.prod(X_train.shape):.4%})")

訓練データ行列の形状: (810, 8091) (非ゼロ率 = 5.0734%)


文字 1〜3-gram の TF-IDF 行列は**8,000 次元超でありながら非ゼロ率が約 5% しかない「高次元スパース」**（書籍 p.55 脚注 7）な表現になります。

---

## 1. SVM：線形カーネル vs 多項式カーネル vs RBF カーネル（2-1-2、p.55）

書籍 p.55 では、高次元スパースになりやすい自然言語処理のデータでは、非線形な **RBF カーネル** や高次の多項式カーネルよりも **線形カーネル（`liblinear` 等）** が高速で扱いやすく、重みの解釈もしやすいことが解説されています。実際に精度・過学習（訓練 vs テストの差）・処理時間を比較します。

In [3]:
print("テスト文書ID先頭5件 (ch1_tfidf_baseline.ipynb・ch2_bert.ipynb と同一分割):", test_df["文書ID"].head(5).tolist())

models = [
    ("線形カーネル (LinearSVC, C=1.0)", LinearSVC(C=1.0, random_state=42)),
    ("多項式カーネル 2次 (SVC poly d=2, C=2.0)", SVC(kernel="poly", degree=2, C=2.0, random_state=42)),
    ("多項式カーネル 3次 (SVC poly d=3, C=2.0)", SVC(kernel="poly", degree=3, C=2.0, random_state=42)),
    ("RBF カーネル (SVC rbf, C=1.0)", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)),
    ("RBF カーネル 強適合 (SVC rbf, C=10, gamma=5.0)", SVC(kernel="rbf", C=10.0, gamma=5.0, random_state=42)),
]

svm_rows = []
for name, clf in models:
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    pred_tr = clf.predict(X_train)
    pred_dv = clf.predict(X_dev)
    pred_te = clf.predict(X_test)
    elapsed = (time.perf_counter() - t0) * 1000
    svm_rows.append({
        "手法": name,
        "訓練Acc": round(accuracy_score(y_train, pred_tr), 3),
        "開発Acc": round(accuracy_score(y_dev, pred_dv), 3),
        "テストAcc": round(accuracy_score(y_test, pred_te), 3),
        "テストF1": round(f1_score(y_test, pred_te), 3),
        "学習+推論 [ms]": round(elapsed, 1),
    })

pd.DataFrame(svm_rows)

テスト文書ID先頭5件 (ch1_tfidf_baseline.ipynb・ch2_bert.ipynb と同一分割): [789, 861, 668, 200, 288]


,手法,訓練Acc,開発Acc,テストAcc,テストF1,学習+推論 [ms]
0,"線形カーネル (LinearSVC, C=1.0)",1.0,0.989,0.97,0.970,21.9
1,"多項式カーネル 2次 (SVC poly d=2, C=2.0)",1.0,0.989,0.97,0.969,2563.5
2,"多項式カーネル 3次 (SVC poly d=3, C=2.0)",1.0,0.978,0.97,0.969,2658.8
3,"RBF カーネル (SVC rbf, C=1.0)",1.0,0.989,0.97,0.969,2396.9
4,"RBF カーネル 強適合 (SVC rbf, C=10, gamma=5.0)",1.0,0.889,0.90,0.909,2545.2


- 線形カーネル（`LinearSVC`）はわずか **数十ミリ秒** で動作し、テストでも高い汎化性能（0.97 前後）を出します。
- このデータと設定では、多項式・RBF カーネルの `SVC` は `LinearSVC` より時間がかかり、テスト正解率は上回りませんでした。モデルの実装も異なるため、実行時間の差にはカーネル以外の要因も含まれます。RBF の `C=10`、`gamma=5` の設定は、訓練で高い正解率を得ても開発・テストでは下がる過学習の例です。

---

## 2. $k$ 近傍法（TF-IDF + コサイン類似度）と根拠事例の提示（2-2-2、p.62）

書籍 p.62 では、小規模・簡易的な手法として **TF-IDF ベクトルとコサイン類似度による $k$ 近傍法（k-NN）** が紹介されています。
k-NN は「どの訓練事例と似ていたからこのクラスに分類したのか」という**判断根拠の事例をそのまま確認でき、エラー分析しやすい**のが大きな特長です。

In [4]:
knn_rows = []
for k in [1, 3, 5, 11, 21]:
    t0 = time.perf_counter()
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(X_train, y_train)
    pred_dv = knn.predict(X_dev)
    pred_te = knn.predict(X_test)
    elapsed = (time.perf_counter() - t0) * 1000
    knn_rows.append({
        "k": k,
        "開発Acc": round(accuracy_score(y_dev, pred_dv), 3),
        "テストAcc": round(accuracy_score(y_test, pred_te), 3),
        "テストF1": round(f1_score(y_test, pred_te), 3),
        "処理時間 [ms]": round(elapsed, 1),
    })

display(pd.DataFrame(knn_rows))

# 開発データで最良の k で詳細を確認
best_k = max(knn_rows, key=lambda r: r["開発Acc"])["k"]
knn_best = KNeighborsClassifier(n_neighbors=best_k, metric="cosine").fit(X_train, y_train)
test_preds = knn_best.predict(X_test)

def inspect_neighbors(test_pos, n_show=3):
    row = test_df.iloc[test_pos]
    dists, idxs = knn_best.kneighbors(X_test[test_pos], n_neighbors=n_show)
    print(f"[テスト文書 ID={row['文書ID']}] 正解={row['ラベル']} / k-NN予測={bool(test_preds[test_pos])} | メモ={row['コメント']}")
    print(f"  本文: {row['文書'][:90]}...")
    print("  --- 参照された類似訓練事例 ---")
    for rank, (d, tr_idx) in enumerate(zip(dists[0], idxs[0]), 1):
        tr = train_df.iloc[tr_idx]
        sim = 1.0 - d
        print(f"    {rank}位 (cos={sim:.3f}, ラベル={tr['ラベル']}, メモ={tr['コメント']}): {tr['文書'][:75]}...")
    print()

correct_pos = int(np.where(test_preds == y_test)[0][0])
wrong_pos = int(np.where(test_preds != y_test)[0][0])
inspect_neighbors(correct_pos)
inspect_neighbors(wrong_pos)

,k,開発Acc,テストAcc,テストF1,処理時間 [ms]
0,1,0.922,0.86,0.863,24.6
1,3,0.922,0.94,0.940,22.7
2,5,0.933,0.93,0.932,22.7
3,11,0.900,0.88,0.887,22.3
4,21,0.911,0.87,0.881,22.2


[テスト文書 ID=789] 正解=True / k-NN予測=True | メモ=入院に伴うコロナ感染判明
  本文: 3日前から38度9分の熱と耳の奥の痛みと寒気が出始めたため、近くの呼吸器内科を受診しました。 体調悪化のため入院して点滴を受けていましたが、院内での抗原定量検査の結果、コロナウイル...
  --- 参照された類似訓練事例 ---
    1位 (cos=0.572, ラベル=True, メモ=入院に伴うコロナ感染判明): 今月10日に強い倦怠感と関節痛で目が覚め、熱を測ると38度2分あったため予約を取って夜間救急センターへ行きました。 体調悪化のため入院して点滴を受けて...
    2位 (cos=0.548, ラベル=True, メモ=入院に伴うコロナ感染判明): 最初は単なる疲れかと思っていましたが、昨日の夜になって悪寒と節々の痛みがひどくなり、職場の近くのクリニックへ行って検査を受けました。 体調悪化のため入...
    3位 (cos=0.534, ラベル=True, メモ=入院に伴うコロナ感染判明): 私の場合は、今月に入ってからに体のだるさと筋肉痛が出てから37度5分の熱が続き、念のため市立病院の内科で検査してもらいました。 体調悪化のため入院して...

[テスト文書 ID=31] 正解=False / k-NN予測=True | メモ=入院時のコロナ検査で陰性
  本文: 体調不良により入院することになりました。 入院中には新型コロナウイルスの検査も行われ、陰性の結果が出たことを確認しました。 現在は退院して家で静養している状態です。...
  --- 参照された類似訓練事例 ---
    1位 (cos=0.397, ラベル=True, メモ=入院中にコロナ感染判明): 私は自宅で療養を続けていましたが、 最近になって体力の衰えと共に症状が再び悪化し、 病院へ入院して治療を受けることになりました。 入院中に新型コロナウ...
    2位 (cos=0.341, ラベル=True, メモ=新型コロナによる体調不良・入院): 今回の新型コロナウイルスにより体調を崩したため入院することになりました。 入院中には医師や看護師から熱心に治療を受けており、退院直前まで安静に過ごすよ...
    3位 (cos=0.339, ラベ

誤分類したテスト事例（`ID=31`：入院時の検査で陰性）についても、上位の近傍訓練事例を並べて見ることで「`入院`・`検査`・`新型コロナウイルス` といった表面的な表現の類似に引っ張られ、`陰性` と `感染判明` の違いを区別できずに似た事例が選ばれた」というエラー原因を直感的に分析できます。

---

## 3. ナイーブベイズ分類器とラプラス平滑化（2-1-2・関連論文 p.213）

書籍 2-1 および関連論文（p.213、正誤表も参照）では、単純な確率的生成モデルである **ナイーブベイズ（Naive Bayes）** も文書分類の強力なベースラインになることが述べられています。

セクション 1・2 と同じ `X_train`, `X_dev`, `X_test`, `y_train`, `y_dev`, `y_test`（TF-IDF）に加え、多項ナイーブベイズ本来の入力である頻度カウント（`CountVectorizer`：同じ文字 1〜3-gram, `min_df=2`）でも `MultinomialNB` を学習し、平滑化パラメータ `alpha`（`1.0`: ラプラス平滑化／Add-1、`0.1`、`1e-9`: 平滑化なしに近い極小値）を `LinearSVC(C=1.0, random_state=42)` と比較します。

あわせて確率予測の過信（ゼロ頻度問題）を捉えるため、確率値を出力できるナイーブベイズについて **Log-Loss（交差エントロピー損失）** と、対数尤度比 $\log \frac{P(w \mid \text{TRUE})}{P(w \mid \text{FALSE})}$（`nb.feature_log_prob_[1] - nb.feature_log_prob_[0]`）の上位表現も確認します。

In [5]:
cnt_vec = CountVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2)
X_train_cnt = cnt_vec.fit_transform(train_df["文書"])
X_dev_cnt = cnt_vec.transform(dev_df["文書"])
X_test_cnt = cnt_vec.transform(test_df["文書"])

def evaluate_classifier(name, clf, Xtr, Xdv, Xte):
    clf.fit(Xtr, y_train)
    pred_tr = clf.predict(Xtr)
    pred_dv = clf.predict(Xdv)
    pred_te = clf.predict(Xte)
    ll_dv = round(log_loss(y_dev, clf.predict_proba(Xdv)), 3) if hasattr(clf, "predict_proba") else np.nan
    ll_te = round(log_loss(y_test, clf.predict_proba(Xte)), 3) if hasattr(clf, "predict_proba") else np.nan
    return {
        "手法": name,
        "訓練Acc": round(accuracy_score(y_train, pred_tr), 3),
        "開発Acc": round(accuracy_score(y_dev, pred_dv), 3),
        "テストAcc": round(accuracy_score(y_test, pred_te), 3),
        "テストF1": round(f1_score(y_test, pred_te), 3),
        "開発LogLoss": ll_dv,
        "テストLogLoss": ll_te,
    }

configs = [
    ("LinearSVC (TF-IDF, C=1.0)", LinearSVC(C=1.0, random_state=42), X_train, X_dev, X_test),
    ("MultinomialNB (TF-IDF, alpha=1.0 / ラプラス)", MultinomialNB(alpha=1.0), X_train, X_dev, X_test),
    ("MultinomialNB (TF-IDF, alpha=0.1)", MultinomialNB(alpha=0.1), X_train, X_dev, X_test),
    ("MultinomialNB (TF-IDF, alpha=1e-9 / 極小)", MultinomialNB(alpha=1e-9), X_train, X_dev, X_test),
    ("MultinomialNB (Count, alpha=1.0 / ラプラス)", MultinomialNB(alpha=1.0), X_train_cnt, X_dev_cnt, X_test_cnt),
    ("MultinomialNB (Count, alpha=0.1)", MultinomialNB(alpha=0.1), X_train_cnt, X_dev_cnt, X_test_cnt),
    ("MultinomialNB (Count, alpha=1e-9 / 極小)", MultinomialNB(alpha=1e-9), X_train_cnt, X_dev_cnt, X_test_cnt),
]
nb_df = pd.DataFrame([evaluate_classifier(*cfg) for cfg in configs])
display(nb_df)

def top_llr_table(vectorizer, Xtr, alpha, n=10):
    nb = MultinomialNB(alpha=alpha).fit(Xtr, y_train)
    llr = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
    feats = np.array(vectorizer.get_feature_names_out())
    top_pos = np.argsort(llr)[::-1][:n]
    top_neg = np.argsort(llr)[:n]
    return pd.DataFrame({
        "TRUE寄り表現": feats[top_pos],
        "対数尤度比(+)": np.round(llr[top_pos], 3),
        "FALSE寄り表現": feats[top_neg],
        "対数尤度比(-)": np.round(llr[top_neg], 3),
    })

print("\n[TF-IDF × MultinomialNB (alpha=1.0) の対数尤度比 上位10表現]")
display(top_llr_table(vec, X_train, alpha=1.0))

print("\n[頻度 Count × MultinomialNB (alpha=1.0 vs 1e-9) の対数尤度比 上位10表現]")
display(top_llr_table(cnt_vec, X_train_cnt, alpha=1.0))
display(top_llr_table(cnt_vec, X_train_cnt, alpha=1e-9))

,手法,訓練Acc,開発Acc,テストAcc,テストF1,開発LogLoss,テストLogLoss
0,"LinearSVC (TF-IDF, C=1.0)",1.000,0.989,0.97,0.970,NaN,NaN
1,"MultinomialNB (TF-IDF, alpha=1.0 / ラプラス)",0.932,0.867,0.86,0.877,0.238,0.275
2,"MultinomialNB (TF-IDF, alpha=0.1)",0.991,0.944,0.96,0.961,0.114,0.086
3,"MultinomialNB (TF-IDF, alpha=1e-9 / 極小)",1.000,0.978,0.99,0.990,0.151,0.079
4,"MultinomialNB (Count, alpha=1.0 / ラプラス)",0.991,0.956,0.96,0.961,0.421,0.448
5,"MultinomialNB (Count, alpha=0.1)",0.998,0.978,0.97,0.970,0.469,0.130
6,"MultinomialNB (Count, alpha=1e-9 / 極小)",1.000,0.978,0.98,0.980,0.801,0.543



[TF-IDF × MultinomialNB (alpha=1.0) の対数尤度比 上位10表現]


,TRUE寄り表現,対数尤度比(+),FALSE寄り表現,対数尤度比(-)
0,わか,1.831,だと,-1.376
1,がわ,1.761,も受け,-1.374
2,がわか,1.761,も受,-1.374
3,かりま,1.721,査も受,-1.355
4,とがわ,1.714,た際、,-1.336
5,わかり,1.714,た際,-1.336
6,染して,1.641,した際,-1.336
7,いるこ,1.410,たが結,-1.336
8,のの,1.379,診した,-1.336
9,たもの,1.379,が結果,-1.336



[頻度 Count × MultinomialNB (alpha=1.0 vs 1e-9) の対数尤度比 上位10表現]


,TRUE寄り表現,対数尤度比(+),FALSE寄り表現,対数尤度比(-)
0,たもの,4.075,だと,-4.130
1,たも,4.075,も受け,-4.112
2,ものの,4.075,も受,-4.112
3,のの,4.075,査も受,-4.095
4,の、,3.819,した際,-4.077
5,のの、,3.819,際、念,-4.077
6,査で新,3.819,際、,-4.077
7,いたこ,3.798,た際、,-4.077
8,ったも,3.755,た際,-4.077
9,判定,3.755,が結,-4.077


,TRUE寄り表現,対数尤度比(+),FALSE寄り表現,対数尤度比(-)
0,たも,24.780,だと,-24.838
1,のの,24.780,も受け,-24.820
2,ものの,24.780,も受,-24.820
3,たもの,24.780,査も受,-24.803
4,の、,24.519,が結,-24.785
5,のの、,24.519,が結果,-24.785
6,査で新,24.519,た際、,-24.785
7,いたこ,24.498,た際,-24.785
8,ったも,24.453,崩して,-24.785
9,判定,24.453,際、,-24.785


- **TF-IDF と頻度カウントでの $\alpha$ の効き方の違い**: L2 正規化された TF-IDF（`X_train`）は 1 文書あたりの合計重みが小さいため、`alpha=1.0` を語彙 8,091 次元すべてに加えると過度な平滑化になり開発・テスト精度が下がります（開発 `0.867`・テスト `0.860`）。一方、本来の多項モデルである頻度カウント（`CountVectorizer`）では `alpha=1.0`（ラプラス平滑化）でも高い精度（開発 `0.956`・テスト `0.960`）が出ます。
- **`alpha=1e-9`（平滑化ほぼなし）の挙動とゼロ頻度問題**: 本データでは文字 1〜3-gram（`min_df=2`）が 1 文書に多数含まれるため、`alpha=1e-9` でも正解率自体は高く保たれます（テスト `0.980`〜`0.990`）。しかし、訓練データの片方のクラスに 0 回しか現れなかった表現（「`たもの`」「`のの`」は TRUE のみ、「`だと`」「`も受け`」は FALSE のみ）に着目すると、`alpha=1.0`（Count）では対数尤度比が `±4.1` 程度に抑えられるのに対し、`alpha=1e-9` では **`±24.8`** まで極端に跳ね上がります。その結果、予測確率が `0` や `1` に張り付いて確率予測の過信を招き、Count 版の `開発LogLoss` は `0.421`（`alpha=1.0`）から `0.801`（`alpha=1e-9`）へ悪化します。
- **生成モデル（ナイーブベイズ）vs 識別モデル（SVM）**: ナイーブベイズは各クラス $c$ の単語確率 $P(w \mid c)$ を独立に数え上げる**生成モデル**であり、集計だけで高速に学習でき、対数尤度比 $\log \frac{P(w \mid \text{TRUE})}{P(w \mid \text{FALSE})}$ として各表現のクラス寄与を直接解釈できます。ただし「`わか`」「`がわ`」「`がわか`」のような重複 n-gram も独立とみなして加算するため確率が振れやすい特徴があります。これに対し `LinearSVC` などの**識別モデル**はクラス境界を直接最適化して相関のある特徴量の重みを調整するため、開発データで安定して高い汎化性能（`0.989`）を示します（書籍 2-1・関連論文 p.213）。